In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
words = open("files/names.txt").read().splitlines()
words[:5]

['emma', 'olivia', 'ava', 'isabella', 'sophia']

In [3]:
len(words)

32033

In [4]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0 
itos = {i:s for s,i in stoi.items()}

In [5]:
block_size = 3 # context lenght: how many characters do we take to predict the next one     
X,Y = [], []
for w in words[:5]:

    print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(context)
        print(''.join(itos[i] for i in context), "--->",itos[ix])
        context = context[1:] + [ix] # crop and update
        
X = torch.tensor(X)
Y = torch.tensor(Y)


emma
[0, 0, 0]
... ---> e
[0, 0, 5]
..e ---> m
[0, 5, 13]
.em ---> m
[5, 13, 13]
emm ---> a
[13, 13, 1]
mma ---> .
olivia
[0, 0, 0]
... ---> o
[0, 0, 15]
..o ---> l
[0, 15, 12]
.ol ---> i
[15, 12, 9]
oli ---> v
[12, 9, 22]
liv ---> i
[9, 22, 9]
ivi ---> a
[22, 9, 1]
via ---> .
ava
[0, 0, 0]
... ---> a
[0, 0, 1]
..a ---> v
[0, 1, 22]
.av ---> a
[1, 22, 1]
ava ---> .
isabella
[0, 0, 0]
... ---> i
[0, 0, 9]
..i ---> s
[0, 9, 19]
.is ---> a
[9, 19, 1]
isa ---> b
[19, 1, 2]
sab ---> e
[1, 2, 5]
abe ---> l
[2, 5, 12]
bel ---> l
[5, 12, 12]
ell ---> a
[12, 12, 1]
lla ---> .
sophia
[0, 0, 0]
... ---> s
[0, 0, 19]
..s ---> o
[0, 19, 15]
.so ---> p
[19, 15, 16]
sop ---> h
[15, 16, 8]
oph ---> i
[16, 8, 9]
phi ---> a
[8, 9, 1]
hia ---> .


In [6]:
X[0],Y[0]

(tensor([0, 0, 0]), tensor(5))

In [7]:
X.shape,X.dtype,Y.shape,Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [8]:
C = torch.randn((27,2))

In [9]:
C[5]

tensor([-0.0919,  0.1413])

In [10]:
F.one_hot(torch.tensor(5),27).float() @ C

tensor([-0.0919,  0.1413])

In [11]:
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

In [12]:
W1 = torch.randn((6,100))
b1 = torch.randn(100)

In [13]:
emb * W1 + b1

RuntimeError: The size of tensor a (2) must match the size of tensor b (100) at non-singleton dimension 2

In [ ]:
torch.cat([emb[:,0,:],emb[:,1,:],emb[:,2,:]],1).shape

torch.Size([32, 6])

In [16]:
len(torch.unbind(emb,1))

3

In [ ]:
torch.cat(torch.unbind(emb,1),1).shape

torch.Size([32, 6])

In [ ]:
emb.view(32,6) == torch.cat(torch.unbind(emb,1),1)

tensor([True, True, True, True, True, True])

In [ ]:
class MyList:

    def __init__(self, *args):
        self.list = list(args)

    def __repr__(self):
        return str(self.list)

    def view(self, rows, cols):

        if rows == -1:
            rows = len(self.list) // cols

        elif cols == -1:
            cols = len(self.list) // rows

        if rows * cols != len(self.list):
            raise ValueError("Invalid shape")

        result = []

        for i in range(rows):
            start = i * cols
            end = start + cols
            result.append(self.list[start:end])

        return result
    


ImportError: attempted relative import with no known parent package